# Alba MiniLLM: Suy luận Logic + Coding + Tool Calling Benchmark

Notebook kiểm chứng trực tiếp kiến trúc **`src.model.MiniLLM`** đã tích hợp đầy đủ:
- `pre_norm`: Residual chuẩn `x + u`
- `gor_v2`: Zero-param GOR-v2 tối ưu đại số (`(1 + coeff) * x + u`)
- `dual_bank_gor_v2`: Dual-Bank bộ nhớ $\mathcal{O}(1)$ + GOR-v2
- `attn_res_block`: Block AttnRes (`N = 8`)
- `attn_res_gor_v2`: **Block AttnRes + GOR-v2 chọn lọc ở Attention** (khuyến nghị #1)
- `attn_res_gor_v2_moe`: **Block AttnRes + GOR-v2 kết hợp `OrthogonalSubMoE`**

Đo tách riêng **BPB & PPL** trên 4 mặt trận: **Logic (`GSM8K + ARC`)**, **Coding (`MBPP + Python-Alpaca`)**, **Tool Calling (`JSON schema + Multi-turn Recovery`)**, và **OOD (`MMLU Formal Logic / Math + HumanEval`)**.

In [ ]:
import os
import torch

os.system("nvidia-smi")
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

os.system("pip install -q datasets transformers matplotlib")

In [ ]:
import os
import subprocess

repo_dir = "/content/Alba-model"
if os.path.exists(repo_dir):
    subprocess.run(["rm", "-rf", repo_dir], check=True)
subprocess.run(
    ["git", "clone", "-b", "Khang", "https://github.com/potatips/Alba-model.git", repo_dir],
    check=True,
)
%cd /content/Alba-model
os.system("python3 -m unittest gor_research.test_gor_research -v")

In [ ]:
%cd /content/Alba-model
!python3 -m gor_research.alba_reasoning_study --device cuda --output-dir /content/alba_reasoning_results

In [ ]:
import json
from IPython.display import Image, display

res_dir = "/content/alba_reasoning_results"
with open(f"{res_dir}/alba_reasoning_results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("META:", json.dumps(data["meta"]))
print("\n=== ALBA MINILLM: LOGIC / CODE / TOOL / OOD ===")
for k, v in data["results"].items():
    print(
        f"{k:20s} | ValBPB={v['val_bpb']:.4f} | Logic={v['logic_bpb']:.4f} | "
        f"Code={v['code_bpb']:.4f} | Tool={v['tool_bpb']:.4f} | OOD={v['ood_bpb']:.4f} | "
        f"Active={v['active_params']:,} | {v['tok_per_sec']:,.0f} tok/s"
    )

display(Image(filename=f"{res_dir}/alba_reasoning_report.png"))

In [ ]:
from google.colab import files

res_dir = "/content/alba_reasoning_results"
files.download(f"{res_dir}/alba_reasoning_results.json")
files.download(f"{res_dir}/alba_reasoning_report.png")
files.download(f"{res_dir}/alba_reasoning_report.html")